In [1]:
# Standard Library
import datetime
import io
import os
import random
import sys
import warnings

from datetime import datetime, timedelta
from pathlib import Path

# Data Handling
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter, FuncFormatter, MultipleLocator

# Data Sources
import yfinance as yf
import pandas_datareader.data as web

# Statistical Analysis
import statsmodels.api as sm

# Machine Learning
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Suppress warnings
warnings.filterwarnings("ignore")

In [2]:
# Add the source subdirectory to the system path to allow import config from settings.py
current_directory = Path(os.getcwd())
website_base_directory = current_directory.parent.parent.parent
src_directory = website_base_directory / "src"
sys.path.append(str(src_directory)) if str(src_directory) not in sys.path else None

# Import settings.py
from settings import config

# Add configured directories from config to path
SOURCE_DIR = config("SOURCE_DIR")
sys.path.append(str(Path(SOURCE_DIR))) if str(Path(SOURCE_DIR)) not in sys.path else None

# Add other configured directories
BASE_DIR = config("BASE_DIR")
CONTENT_DIR = config("CONTENT_DIR")
POSTS_DIR = config("POSTS_DIR")
PAGES_DIR = config("PAGES_DIR")
PUBLIC_DIR = config("PUBLIC_DIR")
SOURCE_DIR = config("SOURCE_DIR")
DATA_DIR = config("DATA_DIR")
DATA_MANUAL_DIR = config("DATA_MANUAL_DIR")

# Print system path
for i, path in enumerate(sys.path):
    print(f"{i}: {path}")

0: /usr/lib/python313.zip
1: /usr/lib/python3.13
2: /usr/lib/python3.13/lib-dynload
3: 
4: /home/jared/python-virtual-envs/general_313/lib/python3.13/site-packages
5: /home/jared/python-virtual-envs/general_313/lib/python3.13/site-packages/setuptools/_vendor
6: /home/jared/Cloud_Storage/Dropbox/src
7: /home/jared/Cloud_Storage/Dropbox/Websites/jaredszajkowski.github.io_congo/src


In [8]:
# from calc_fed_cycle_asset_performance import calc_fed_cycle_asset_performance
# from df_info import df_info
# from df_info_markdown import df_info_markdown
# from export_track_md_deps import export_track_md_deps
# from load_data import load_data
# from pandas_set_decimal_places import pandas_set_decimal_places
# from plot_bar_returns_ffr_change import plot_bar_returns_ffr_change
# from plot_timeseries import plot_timeseries
# from plot_scatter_regression_ffr_vs_returns import plot_scatter_regression_ffr_vs_returns
# from sm_ols_summary_markdown import sm_ols_summary_markdown
# from summary_stats import summary_stats
# from yf_pull_data import yf_pull_data
from ndl_pull_data import ndl_pull_data

In [9]:
import nasdaqdatalink
import numpy as np
import os
import pandas as pd

from IPython.display import display
from load_api_keys import load_api_keys
from pathlib import Path
from settings import config

# Load API keys from the environment
api_keys = load_api_keys()

# Get the environment variable for where data is stored
DATA_DIR = config("DATA_DIR")

def ndl_pull_data_2(
    base_directory,
    ticker: str,
    source: str,
    asset_class: str,
    excel_export: bool,
    pickle_export: bool,
    output_confirmation: bool,
) -> pd.DataFrame:
    
    """
    Download daily price ata from Nasdaq Data Link and add many missing columns and export it.

    Parameters:
    -----------
    base_directory
        Root path to store downloaded data.
    ticker : str
        Ticker symbol to download.
    source : str
        Name of the data source (e.g., 'Nasdaq_Data_Link').
    asset_class : str
        Asset class name (e.g., 'Equities').
    excel_export : bool
        If True, export data to Excel format.
    pickle_export : bool
        If True, export data to Pickle format.
    output_confirmation : bool
        If True, print confirmation message.

    Returns:
    --------
    df : pd.DataFrame
        DataFrame containing the downloaded data.
    """

    # Command to pull data
    # If start date and end date are not specified the entire data set is included
    df = nasdaqdatalink.get_table("NDAQ/USEDHADJ", symbol=ticker, paginate=True, api_key=api_keys["NASDAQ_DATA_LINK_KEY"])

    # Sort columns by date ascending
    df.sort_values('date', ascending = True, inplace = True)

    # # Rename the date column
    # df.rename(columns = {'date':'Date'}, inplace = True)

    # # Set index to date column
    # df.set_index('Date', inplace = True)

    # # Replace all split values of 1.0 with NaN
    # df['split'] = df['split'].replace(1.0, np.nan)

    # # Create a new data frame with split values only
    # df_splits = df.drop(columns = {'ticker', 'open', 'high', 'low', 
    #                                'close', 'volume', 'dividend', 
    #                                'adj_open', 'adj_high', 
    #                                'adj_low', 'adj_close', 
    #                                'adj_volume'}).dropna()

    # # Create a new column for cumulative split
    # df_splits['Cum_Split'] = df_splits['split'].cumprod()

    # # Drop original split column before combining dataframes
    # df_splits.drop(columns = {'split'}, inplace = True)

    # # Merge df and df_split dataframes
    # df_comp = pd.merge(df, df_splits, on='Date', how="outer")

    # # Forward fill for all cumulative split values
    # df_comp['Cum_Split'] = df_comp['Cum_Split'].ffill()

    # # Replace all split and cumulative split values of NaN with 1.0 to have complete split values
    # df_comp['split'] = df_comp['split'].replace(np.nan, 1.0)
    # df_comp['Cum_Split'] = df_comp['Cum_Split'].replace(np.nan, 1.0)

    # # Calculate the non adjusted prices based on the splits only
    # df_comp['non_adj_open_split_only'] = df_comp['open'] * df_comp['Cum_Split']
    # df_comp['non_adj_high_split_only'] = df_comp['high'] * df_comp['Cum_Split']
    # df_comp['non_adj_low_split_only'] = df_comp['low'] * df_comp['Cum_Split']
    # df_comp['non_adj_close_split_only'] = df_comp['close'] * df_comp['Cum_Split']
    # df_comp['non_adj_dividend_split_only'] = df_comp['dividend'] * df_comp['Cum_Split']

    # # Calculate the adjusted prices based on the splits
    # df_comp['Open'] = df_comp['non_adj_open_split_only'] / df_comp['Cum_Split'][-1]
    # df_comp['High'] = df_comp['non_adj_high_split_only'] / df_comp['Cum_Split'][-1]
    # df_comp['Low'] = df_comp['non_adj_low_split_only'] / df_comp['Cum_Split'][-1]
    # df_comp['Close'] = df_comp['non_adj_close_split_only'] / df_comp['Cum_Split'][-1]
    # df_comp['Dividend'] = df_comp['non_adj_dividend_split_only'] / df_comp['Cum_Split'][-1]
    # df_comp['Dividend_Pct_Orig'] = df_comp['dividend'] / df_comp['close']
    # df_comp['Dividend_Pct_Adj'] = df_comp['Dividend'] / df_comp['Close']

    # # Create directory
    # directory = f"{base_directory}/{source}/{asset_class}/Daily"
    # os.makedirs(directory, exist_ok=True)

    # # Export to excel
    # if excel_export == True:
    #     df_comp.to_excel(f"{directory}/{ticker}.xlsx", sheet_name="data")
    # else:
    #     pass

    # # Export to pickle
    # if pickle_export == True:
    #     df_comp.to_pickle(f"{directory}/{ticker}.pkl")
    # else:
    #     pass

    # # Output confirmation
    # if output_confirmation == True:
    #     print(f"The first and last date of data for {ticker} is: ")
    #     display(df[:1])
    #     display(df[-1:])
    #     print(f"NDL data complete for {ticker}")
    #     print(f"--------------------")
    # else:
    #     pass

    df_comp = df

    return df_comp

if __name__ == "__main__":

    df = ndl_pull_data_2(
        base_directory=DATA_DIR,
        ticker="TLT",
        source="Nasdaq_Data_Link",
        asset_class="Exchange_Traded_Funds",
        excel_export=True,
        pickle_export=True,
        output_confirmation=True,
    )

In [10]:
df.head()

,date,symbol,symbol_new,composite_figi,share_class_figi,volume,low,high,open,close,volume_adj,low_adj,high_adj,open_adj,close_adj,action,value
None,,,,,,,,,,,,,,,,,
515,2014-01-02,TLT,TLT,BBG000BJKYW3,BBG001S8MLN3,8580808.0,101.69,102.39,101.72,102.17,8580808.0,73.508,74.014,73.530,73.855,None,None
2201,2014-01-03,TLT,TLT,BBG000BJKYW3,BBG001S8MLN3,4083831.0,101.76,102.45,101.81,102.17,4083831.0,73.559,74.058,73.595,73.855,None,None
1548,2014-01-06,TLT,TLT,BBG000BJKYW3,BBG001S8MLN3,7796228.0,102.37,103.00,102.38,102.60,7796228.0,74.000,74.455,74.007,74.166,None,None
1905,2014-01-07,TLT,TLT,BBG000BJKYW3,BBG001S8MLN3,4428082.0,102.56,102.99,102.82,102.86,4428082.0,74.137,74.448,74.325,74.354,None,None
615,2014-01-08,TLT,TLT,BBG000BJKYW3,BBG001S8MLN3,8512987.0,102.10,102.69,102.45,102.58,8512987.0,73.805,74.231,74.058,74.152,None,None


In [12]:
df_old = ndl_pull_data(
    base_directory=DATA_DIR,
    ticker="TQQQ",
    source="Nasdaq_Data_Link",
    asset_class="Exchange_Traded_Funds",
    excel_export=True,
    pickle_export=True,
    output_confirmation=True,
)

df_old.head()


The first and last date of data for TQQQ is: 


,ticker,open,high,low,close,volume,dividend,split,adj_open,adj_high,...,non_adj_low_split_only,non_adj_close_split_only,non_adj_dividend_split_only,Open,High,Low,Close,Dividend,Dividend_Pct_Orig,Dividend_Pct_Adj
Date,,,,,,,,,,,,,,,,,,,,,
2010-02-11,TQQQ,78.12,83.5,77.87,83.050003,288000.0,0.0,1.0,0.392517,0.419549,...,77.87,83.050003,0.0,0.406875,0.434896,0.405573,0.432552,0.0,0.0,0.0


,ticker,open,high,low,close,volume,dividend,split,adj_open,adj_high,...,non_adj_low_split_only,non_adj_close_split_only,non_adj_dividend_split_only,Open,High,Low,Close,Dividend,Dividend_Pct_Orig,Dividend_Pct_Adj
Date,,,,,,,,,,,,,,,,,,,,,
2025-05-29,TQQQ,73.0,73.0596,69.33,70.37,97717242.0,0.0,1.0,73.0,73.0596,...,13311.36,13511.04,0.0,73.0,73.0596,69.33,70.37,0.0,0.0,0.0


NDL data complete for TQQQ
--------------------


,ticker,open,high,low,close,volume,dividend,split,adj_open,adj_high,...,non_adj_low_split_only,non_adj_close_split_only,non_adj_dividend_split_only,Open,High,Low,Close,Dividend,Dividend_Pct_Orig,Dividend_Pct_Adj
Date,,,,,,,,,,,,,,,,,,,,,
2010-02-11,TQQQ,78.12,83.50,77.87,83.050003,288000.0,0.0,1.0,0.392517,0.419549,...,77.87,83.050003,0.0,0.406875,0.434896,0.405573,0.432552,0.0,0.0,0.0
2010-02-12,TQQQ,80.79,84.11,80.32,83.389999,716800.0,0.0,1.0,0.405933,0.422614,...,80.32,83.389999,0.0,0.420781,0.438073,0.418333,0.434323,0.0,0.0,0.0
2010-02-16,TQQQ,85.35,86.82,84.01,86.620003,801600.0,0.0,1.0,0.428845,0.436231,...,84.01,86.620003,0.0,0.444531,0.452187,0.437552,0.451146,0.0,0.0,0.0
2010-02-17,TQQQ,87.78,88.11,86.46,88.089996,1598400.0,0.0,1.0,0.441054,0.442712,...,86.46,88.089996,0.0,0.457188,0.458906,0.450312,0.458802,0.0,0.0,0.0
2010-02-18,TQQQ,88.00,90.29,87.47,89.760002,3238400.0,0.0,1.0,0.442160,0.453666,...,87.47,89.760002,0.0,0.458333,0.470260,0.455573,0.467500,0.0,0.0,0.0
